# マイクログリッド設計 + 複数代表日運用の同時決定 — Microgrid Design & Operation

オフグリッド化を検討する工業団地のマイクログリッド設計者は、PV・蓄電池・非常用発電機の
**設備容量**(サイジング、投資決定)と、季節を代表する複数の代表日(夏平日・冬平日・
中間期・週末)にわたる**充放電・出力配分**(運用決定)を同時に決めなければならない。
容量を過小に見積もれば特定の代表日で運用が破綻し、過大に見積もれば投資回収が悪化する。

この問題を特に難しくしているのは、蓄電池の内部抵抗損失が
`loss * cap_batt >= k * (充放電出力)^2` という**双曲線**で表され、**容量という設計変数
そのものが非線形項に現れる**ことである。容量と運用を分離すると、この物理的な結合
(「大きく作るほど同じ出力でも効率が良いが投資がかさむ」)自体が消えてしまう。

1. **素朴な定式化** — サイジング変数・双曲線制約の所在を確認
2. **診断** — `mk.analyze` で観測し、どの症状が発火するかを見る
3. **診断の中身を見る** — ルートLP緩和での非線形制約の違反、数値スケールの発火根拠を掘る
4. **改善を試す** — numerical_scale findingsのrecipeを試し、効果を正直に測る

題材: `samples/energy_and_microgrid/microgrid_design_operation.py`

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import microgrid_design_operation as mdo
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

`build_model("default")` は代表日4×時刻14の規模。設計変数(`cap_pv`・`cap_batt` は連続、
`n_gen` は整数)は全代表日で共有され、運用変数は代表日×時刻ごとに独立して持つ。

In [2]:
m = mdo.build_model("default")
nDay, nT = m.data["dims"]
n_int = sum(1 for v in m.getVars() if v.vtype() in ("INTEGER", "BINARY"))
n_cont = sum(1 for v in m.getVars() if v.vtype() == "CONTINUOUS")
n_nonlin = sum(1 for c in m.getConss() if not c.isLinear())
print(f"代表日={nDay} 時刻={nT}")
print(f"変数: 整数(発電機台数) {n_int} 個 / 連続(サイジング+運用) {n_cont} 個")
print(f"制約: {m.getNConss()} 本(内 非線形 = {n_nonlin} 本 = 代表日×時刻ぶんの蓄電池損失式)")
print(f"BATT_K_LOSS={mdo.BATT_K_LOSS}  蓄電池容量範囲=[{5.0},{1500.0}]")
print("非線形制約の例(1本):", "loss*cap_batt >= BATT_K_LOSS*(p_charge+p_discharge)^2")

代表日=4 時刻=14
変数: 整数(発電機台数) 57 個 / 連続(サイジング+運用) 398 個
制約: 512 本(内 非線形 = 56 本 = 代表日×時刻ぶんの蓄電池損失式)
BATT_K_LOSS=0.22  蓄電池容量範囲=[5.0,1500.0]
非線形制約の例(1本): loss*cap_batt >= BATT_K_LOSS*(p_charge+p_discharge)^2


## 2. 診断 (`mk.analyze`)

In [3]:
report = mk.analyze(lambda: mdo.build_model("default"), name="microgrid-default", time_limit=30)
print(report.summary())
print()
print({k: report.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "has_nonlinear",
    "bottleneck_type", "bottleneck_rel_viol",
    "residual_coef_ratio", "coef_ratio", "residual_bigm_count"]})

[microgrid-default] 検出症状 1件:
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化

{'gap': 0.0013538130222004385, 'nodes': 1, 'nsols': 1, 'spatial_share': 0.0, 'has_nonlinear': True, 'bottleneck_type': 'batt_loss_2', 'bottleneck_rel_viol': 0.1478875610160386, 'residual_coef_ratio': 319783.3774117384, 'coef_ratio': 319783.3774117384, 'residual_bigm_count': 10}


`has_nonlinear=True`(双曲線制約を持つ真の非凸MINLP)にもかかわらず、`spatial_share=0`・
`nodes=1` — 空間分枝を一切使わずルートで解けている。診断で発火するのは**数値スケール**
(`numerical_scale`)のみで、非凸そのものは症状として現れない。3節でその理由(なぜこの
非線形制約が「難しくない」のか)と、数値スケールの発火根拠の両方を掘り下げる。

## 3. 診断の中身を見る

### 3-1. なぜ「非凸」なのに空間分枝ゼロなのか — ルートLP緩和の違反を見る

`collect_root_violations` はルートLP緩和解を非線形制約に代入し、実際の違反量を測る。
非線形項が本当に「効いていない」なら違反はゼロに近いはずである。

In [4]:
m3 = mdo.build_model("default")
vdf = collect_root_violations(m3)
by_type = violation_by_type(vdf)
print(by_type.head(8).to_string(index=False))

        ctype     mean_rel      max_rel  n
  batt_loss_2 1.478876e-01 3.490284e-01 14
  batt_loss_1 1.478876e-01 3.490284e-01 14
  batt_loss_3 9.450573e-02 3.490284e-01 14
  batt_loss_0 8.801345e-02 3.490284e-01 14
     pv_cap_0 4.736952e-15 5.684342e-14 12
soc_balance_2 7.930164e-18 1.110223e-16 14
soc_balance_0 3.965082e-18 5.551115e-17 14
    balance_0 0.000000e+00 0.000000e+00 14


`batt_loss_*` 制約はルートLP緩和で平均10-35%もの相対違反を持つ — 非線形項は
「効いていない」どころか、線形緩和からは大きく逸脱している。それでも探索木が
根ノードだけで済むのは、**この制約が非凸ではなく凸だから**である。

`loss * cap_batt >= k * p^2`(`p = p_charge + p_discharge`, `cap_batt > 0`)は、
`loss >= k * p^2 / cap_batt` と同値であり、これは凸2次関数 `k*p^2` の
**perspective(遠近)関数**の下側エピグラフである。perspective変換は凸性を保存するため、
`cap_batt` が下限5(>0)で抑えられている限り、この制約は `(p, cap_batt, loss)` の
関数として**結合凸**になる。SCIPはこの種の二次制約(rotated second-order cone相当)を
線形近似のカット(外部近似)で締めることができ、整数×連続の空間分枝を要しない
——「容量という設計変数が非線形項に現れる」という結合の強さと、「その非線形が
探索を難しくするか」は別軸であることが分かる(`minlpkit.perspective_quadratic` が
半連続の on/off 版として同じ変換を提供しているのもこのため)。

### 3-2. 数値スケール — `residual_coef_ratio` の発火根拠

`n_gen` の投資費 `COST_GEN_UNIT=55000`[$]という金額スケールが、他の物理量(kW・比率)
と桁違いなことが `residual_coef_ratio` を押し上げている。presolve後の最小・最大の
出所を実際に確認する。

In [5]:
m4 = mdo.build_model("default")
m4.hideOutput()
pre = residual_scale(m4)
print(f"presolve後: min={pre['min']:.3g} max={pre['max']:.3g} ratio={pre['ratio']:.3g}")

m5 = mdo.build_model("default")
m5.hideOutput(); m5.presolve()
df_coef = extract_coefficients(m5)
df_sorted = df_coef.sort_values("magnitude")
print("--- 最小側 ---")
print(df_sorted.head(5)[["source", "magnitude", "name"]].to_string(index=False))
print("--- 最大側 ---")
print(df_sorted.tail(5)[["source", "magnitude", "name"]].to_string(index=False))

presolve後: min=0.172 max=5.5e+04 ratio=3.2e+05
--- 最小側 ---
source  magnitude        name
  制約係数   0.171991 pv_cap_2_12
  制約係数   0.171991  pv_cap_2_1
  制約係数   0.174374  pv_cap_3_1
  制約係数   0.174374 pv_cap_3_12
  制約係数   0.174985 pv_cap_1_12
--- 最大側 ---
source  magnitude      name
  変数境界     2220.0  loss_0_8
  変数境界     2220.0  loss_0_7
  変数境界     2220.0 loss_3_12
  変数境界     2220.0 loss_3_11
  目的係数    55000.0     n_gen


In [6]:
fig = go.Figure(layout=base_layout(
    "presolve後の係数分布(出所別) — 最大は発電機投資費(55,000$/台)、最小は物理量(比率・kW)",
    "出所", "|係数| (log scale)", height=400))
for src, color in [("制約係数", C["s1"]), ("RHS/LHS", C["muted"]), ("目的係数", C["warn"]),
                   ("変数境界", C["s2"])]:
    sub = df_coef[df_coef["source"] == src]
    if sub.empty:
        continue
    fig.add_trace(go.Box(y=sub["magnitude"], name=src, marker_color=color, boxpoints="outliers"))
fig.update_yaxes(type="log")
show(fig)

最小側は `pv_cap_*` 制約の係数(≈0.17、その代表日・時刻のPV出力比率そのもの
——単位容量あたりの日射プロファイル)、最大側は目的係数 `n_gen` の投資費(55,000$)。
どちらも presolve の丸め誤差ではなく**モデルが本来持つ値**であり、「投資費(ドル)」と
「PV出力比率(無次元・0〜1)」という**単位系の違う量が同じ係数行列に同居している**ことが
`residual_coef_ratio` の実体である。`static_diag.matrix_condition` が焦点を当てる
「Big-Mの緩さ」とは異なり、こちらは**通貨単位と物理単位の桁違いという、この種の
投資×運用モデルに構造的に付随する数値スケール**である。4節でこれを実際に直してみる。

## 4. 改善を試す — 目的関数のスケール変換($ → $k)

`numerical_scale` の recipe は「スケーリング・Big-MのIndicator/SOS化・係数の再定式化」。
ここではBig-Mではなく単位の桁違いが主因なので、投資費を千ドル単位($k)に変換した版
(最適解・最適な投資/運用の意思決定は不変、目的関数の値だけがスケールされる)を作り、
数値スケール指標と実際の求解性能の両方を比較する。

In [7]:
def build_scaled(scale="default"):
    d = mdo._data(scale)
    nDay, nT = d["nDay"], d["nT"]
    pv_profile, demand, day_weight = d["pv_profile"], d["demand"], d["day_weight"]
    K = 1000.0  # 投資費を$→$kへ(残りは物理量のままなので係数レンジが縮む)

    m = Model("Microgrid_Design_Operation_Scaled")
    DAY, T = range(nDay), range(nT)
    cap_pv = m.addVar(vtype="C", lb=0.0, ub=1200.0, name="cap_pv")
    cap_batt = m.addVar(vtype="C", lb=5.0, ub=1500.0, name="cap_batt")
    n_gen = m.addVar(vtype="I", lb=0, ub=mdo.N_GEN_MAX, name="n_gen")
    p_pv, p_gen, p_charge, p_discharge, p_grid, soc, loss = ({} for _ in range(7))
    z_gen = {}
    for dday in DAY:
        for t in T:
            p_pv[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_pv_{dday}_{t}")
            p_gen[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_gen_{dday}_{t}")
            z_gen[dday, t] = m.addVar(vtype="I", lb=0, ub=mdo.N_GEN_MAX, name=f"z_gen_{dday}_{t}")
            p_charge[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_charge_{dday}_{t}")
            p_discharge[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_discharge_{dday}_{t}")
            p_grid[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_grid_{dday}_{t}")
            loss[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"loss_{dday}_{t}")
        for t in range(nT + 1):
            soc[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"soc_{dday}_{t}")
    for dday in DAY:
        m.addCons(soc[dday, 0] == 0.5 * cap_batt)
        m.addCons(soc[dday, nT] >= 0.5 * cap_batt)
        for t in T:
            m.addCons(p_pv[dday, t] <= cap_pv * float(pv_profile[dday, t]))
            m.addCons(z_gen[dday, t] <= n_gen)
            m.addCons(p_gen[dday, t] <= mdo.GEN_UNIT_CAP * z_gen[dday, t])
            m.addCons(p_charge[dday, t] <= mdo.MAX_CRATE * cap_batt)
            m.addCons(p_discharge[dday, t] <= mdo.MAX_CRATE * cap_batt)
            m.addCons(soc[dday, t] <= cap_batt)
            m.addCons(loss[dday, t] * cap_batt >= mdo.BATT_K_LOSS
                      * (p_charge[dday, t] + p_discharge[dday, t])
                      * (p_charge[dday, t] + p_discharge[dday, t]))
            m.addCons(soc[dday, t + 1] == soc[dday, t] + mdo.ETA_C * p_charge[dday, t]
                      - p_discharge[dday, t] / mdo.ETA_D - loss[dday, t])
            m.addCons(p_pv[dday, t] + p_gen[dday, t] + p_discharge[dday, t] + p_grid[dday, t]
                      == float(demand[dday, t]) + p_charge[dday, t])
    capex_k = (mdo.COST_PV / K) * cap_pv + (mdo.COST_BATT / K) * cap_batt \
        + (mdo.COST_GEN_UNIT / K) * n_gen
    opex_k = quicksum(float(day_weight[dday]) / K * (
        mdo.FUEL_COST * p_gen[dday, t] + mdo.GRID_COST * p_grid[dday, t]
    ) for dday in DAY for t in T)
    m.setObjective(capex_k + opex_k, "minimize")
    m.data = dict(cap_pv=cap_pv, cap_batt=cap_batt, n_gen=n_gen)
    return m

pre_scale = scale_summary(extract_coefficients(mdo.build_model("default")))
post_scale = scale_summary(extract_coefficients(build_scaled("default")))
print(f"presolve前(元): ratio={pre_scale['ratio']:.3g}  max={pre_scale['max']:.3g}")
print(f"presolve前($k化): ratio={post_scale['ratio']:.3g}  max={post_scale['max']:.3g}")

presolve前(元): ratio=3.2e+05  max=5.5e+04
presolve前($k化): ratio=5.7e+04  max=1.5e+03


In [8]:
df = mk.compare_variants(
    {"元($単位)": lambda: mdo.build_model("default"),
     "$k単位": build_scaled}, time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,元($単位),514983.123011,514301.777267,0.001354,1
1,$k単位,514.979204,514.301777,0.001354,1


In [9]:
orig_row = df.loc[df["variant"] == "元($単位)"].iloc[0]
scaled_row = df.loc[df["variant"] == "$k単位"].iloc[0]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("係数レンジ比(静的、presolve前)", "最終gap [%]", "探索ノード数"))
labels = ["元($単位)", "$k単位"]
bar_colors = [C["muted"], C["s1"]]
fig.add_trace(go.Bar(x=labels, y=[pre_scale["ratio"], post_scale["ratio"]],
    marker_color=bar_colors, text=[f"{v:.2e}" for v in [pre_scale["ratio"], post_scale["ratio"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=[orig_row["final_gap"] * 100, scaled_row["final_gap"] * 100],
    marker_color=bar_colors, text=[f"{v*100:.2f}%" for v in [orig_row["final_gap"], scaled_row["final_gap"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=[orig_row["nodes"], scaled_row["nodes"]],
    marker_color=bar_colors, text=[int(v) for v in [orig_row["nodes"], scaled_row["nodes"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=3)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="目的関数の単位スケール変換の効果 — 係数レンジは大きく縮むが求解性能は元々十分速い",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

係数レンジ比は`$k`化で大きく縮む(投資費の最大値が1000分の1になるため)。一方
`root_dual`/`final_gap`/`nodes` はどちらもほぼ同じ(すでに root 1ノードでgap0.1%程度)
——**この規模では単位変換の効果は「診断指標をきれいにする」ことに留まり、実際の求解
性能には表れない**。これは3節の「非線形項は凸だから難しくない」という結論と整合的
であり、`08_condition_number.ipynb` で見た「小さい問題はpresolveだけで解けてしまい
数値健全性の効果が測定不能になる」構造とも一致する。

## まとめ

- この問題の核心は「PV/蓄電池/発電機の容量(投資)」と「複数代表日の運用」の同時決定
  であり、蓄電池損失の双曲線 `loss*cap_batt >= k*p^2` が容量という設計変数を非線形項に
  巻き込む——設計と運用を分離すると消えてしまう真の物理結合である。
- しかし**この双曲線は数学的に凸**(2次関数のperspective変換)であり、`cap_batt`の
  下限が正である限り空間分枝を要しない。ルートLP緩和では最大35%の違反があっても、
  SCIPは外部近似カットだけでこれを締められる。「設計変数が非線形項に現れる」という
  結合の強さと、「その非線形が探索を難しくするか」は別の軸であることを実測で示した。
- 発火した`numerical_scale`はBig-Mではなく「投資費(ドル)と運用量(kW・比率)の単位の
  桁違い」が原因で、$k単位への変換で係数レンジは縮むが、この規模では求解性能自体は
  元々十分だったため効果は測定できなかった(正直な結果)。

### この診断が対応する実務の意思決定

マイクログリッド設計者が投資規模を決める際、「容量を増やせば運用効率が上がるが投資が
かさむ」というトレードオフを正しく捉えるには、この非線形結合を保ったまま解く必要がある。
本notebookの診断は「この非線形性は(凸なので)最適性の証明を妨げない」という安心材料と、
「数値スケールの是正は診断上の整理であって、この規模では求解速度には効かない」という
実務的な優先順位付けの両方を与える。

関連: [プレイブック 8. 条件数・数値健全性](../../playbook/08-condition.md) /
サンプルカタログ [エネルギー・マイクログリッド](../../samples/energy_and_microgrid.md)